In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import time
import os

# Kiểm tra thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")
if device.type == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

Đang sử dụng thiết bị: cuda
GPU Model: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
# --- CẤU HÌNH ---
TRAIN_DIR = r"C:\AI_PBL\Dataset\train"
VALID_DIR = r"C:\AI_PBL\Dataset\valid"

BATCH_SIZE = 32         
NUM_EPOCHS = 50         
LEARNING_RATE = 0.01    
IMG_SIZE = 224

In [3]:
print("Đang nạp dữ liệu...")

# Train: Data Augmentation mạnh để chống Overfitting
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    # Chỉnh màu sắc (Quan trọng cho đồ ăn)
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Valid: Giữ nguyên ảnh gốc, chỉ chuẩn hóa
valid_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

try:
    # Load dataset từ thư mục
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
    valid_dataset = datasets.ImageFolder(VALID_DIR, transform=valid_transforms)

    # Tạo DataLoader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Tự động lấy tên lớp và số lượng lớp
    class_names = train_dataset.classes
    NUM_CLASSES = len(class_names)

    print(f"Đã tìm thấy {NUM_CLASSES} class: {class_names}")
    print(f"Số lượng ảnh Train: {len(train_dataset)}")
    print(f"Số lượng ảnh Valid: {len(valid_dataset)}")

except Exception as e:
    print(f"LỖI: {e}")
    print("Vui lòng kiểm tra lại đường dẫn thư mục trong Cell 2.")

Đang nạp dữ liệu...
Đã tìm thấy 18 class: ['backpack', 'belts', 'boots', 'dresses', 'eyewears', 'handbags', 'hatcap', 'jackets', 'jeans', 'perfume', 'phones', 'shirts', 'shorts', 'skirts', 'slides', 'sneakers', 'wallets', 'watch']
Số lượng ảnh Train: 21906
Số lượng ảnh Valid: 5483


In [4]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64
        
        # Stem Block
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # Layers
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Thêm Dropout để giảm Overfitting
        self.dropout = nn.Dropout(p=0.5) 
        
        self.fc = nn.Linear(512*block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.dropout(out) # Kích hoạt dropout
        out = self.fc(out)
        return out

def ResNet18(num_classes):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)

In [5]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0
    start_time = time.time()
    
    print(f"Bắt đầu huấn luyện {num_epochs} epochs...")
    
    for epoch in range(num_epochs):
        # --- TRAINING ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        # Cập nhật Scheduler
        scheduler.step()
        
        epoch_train_loss = running_loss / total
        epoch_train_acc = correct / total
        
        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        
        # Logic Lưu Best Model
        save_msg = ""
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(model.state_dict(), 'best_model_resnet.pth')
            save_msg = "--> ĐÃ LƯU BEST MODEL!"
            
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1}/{num_epochs}] (LR: {current_lr:.5f}) | "
              f"T-Acc: {epoch_train_acc:.4f} | V-Acc: {epoch_val_acc:.4f} {save_msg}")

    time_elapsed = time.time() - start_time
    print(f"\nHoàn tất trong {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Độ chính xác cao nhất (Best Val Acc): {best_acc:.4f}")
    return history

In [ ]:
# 1. Khởi tạo Model
model = ResNet18(num_classes=NUM_CLASSES).to(device)

# 2. Loss Function: Dùng Label Smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 3. Optimizer: Tăng weight decay để giảm overfit
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=1e-3)

# 4. Scheduler: Cosine Annealing
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=0.0001)

# 5. Chạy
history = train_model(model, train_loader, valid_loader, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS)

Bắt đầu huấn luyện 50 epochs...
Epoch [1/50] (LR: 0.00999) | T-Acc: 0.2739 | V-Acc: 0.5238 --> ĐÃ LƯU BEST MODEL!
Epoch [2/50] (LR: 0.00996) | T-Acc: 0.5701 | V-Acc: 0.6367 --> ĐÃ LƯU BEST MODEL!
Epoch [3/50] (LR: 0.00991) | T-Acc: 0.6863 | V-Acc: 0.7572 --> ĐÃ LƯU BEST MODEL!
Epoch [4/50] (LR: 0.00984) | T-Acc: 0.7516 | V-Acc: 0.7886 --> ĐÃ LƯU BEST MODEL!
Epoch [5/50] (LR: 0.00976) | T-Acc: 0.7919 | V-Acc: 0.8211 --> ĐÃ LƯU BEST MODEL!
Epoch [6/50] (LR: 0.00965) | T-Acc: 0.8186 | V-Acc: 0.8191 
Epoch [7/50] (LR: 0.00953) | T-Acc: 0.8395 | V-Acc: 0.8503 --> ĐÃ LƯU BEST MODEL!
Epoch [8/50] (LR: 0.00939) | T-Acc: 0.8552 | V-Acc: 0.8242 
Epoch [9/50] (LR: 0.00923) | T-Acc: 0.8664 | V-Acc: 0.7357 
Epoch [10/50] (LR: 0.00905) | T-Acc: 0.8758 | V-Acc: 0.8776 --> ĐÃ LƯU BEST MODEL!
Epoch [11/50] (LR: 0.00886) | T-Acc: 0.8858 | V-Acc: 0.8065 
Epoch [12/50] (LR: 0.00866) | T-Acc: 0.8974 | V-Acc: 0.8182 
Epoch [13/50] (LR: 0.00844) | T-Acc: 0.8992 | V-Acc: 0.8973 --> ĐÃ LƯU BEST MODEL!
Epoch [1

: 

In [1]:
def plot_results(history):
    acc = history['train_acc']
    val_acc = history['val_acc']
    loss = history['train_loss']
    val_loss = history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(15, 5))

    # Biểu đồ Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.grid(True)

    # Biểu đồ Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title('Training and Validation Loss')
    plt.grid(True)
    plt.show()

plot_results(history)

NameError: name 'history' is not defined

In [3]:
# Hàm dự đoán thử 1 ảnh
from PIL import Image

def predict_image(image_path, model, class_names):
    # Load model tốt nhất
    model.load_state_dict(torch.load('best_model_food10.pth'))
    model.eval()
    
    # Xử lý ảnh
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    image = Image.open(image_path)
    image_tensor = transform(image).unsqueeze(0).to(device) # Thêm batch dimension
    
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted_idx = torch.max(output, 1)
        
    print(f"Dự đoán: {class_names[predicted_idx.item()]}")
    plt.imshow(image)
    plt.show()

# Thay đường dẫn ảnh của bạn vào đây để test
predict_image(r"C:\AI_PBL\Dataset_all\backpack\image_34.jpg", model, class_names)

NameError: name 'model' is not defined